# MoodNote-AI — PhoBERT training + 3-scenario ablation (phase 4, Research Content 2)

Runs on a **Colab T4 GPU**: fine-tunes `vinai/phobert-base-v2` on real_only (UIT-VSMEC), synthetic_only
(accepted synthetic diaries) and combined, all with the same hyperparameters, and tests every run on the
same fixed UIT-VSMEC test set. 7 runs: real_only and combined with seeds 42/43/44, synthetic_only with 42.

Before you start:
- Runtime → Change runtime type → **T4 GPU**.
- Locally run `python -m src.data.ablation` once and **commit + push `data/synthetic/accepted/split.csv`**,
  so Colab trains on exactly the same synthetic split.
- Optional Colab secret `WANDB_API_KEY`; without it W&B logs offline.

Results live on Drive at `MyDrive/MoodNote-AI/ablation/` (`reports/` = one JSON per run + comparison,
`models/` = the seed-42 model of each scenario). If the runtime disconnects, re-run Cell 1 and then the
training cell: finished runs are skipped.

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
import os
import shutil
import subprocess

import torch
from google.colab import drive, userdata

BRANCH = "feature/ToanHuynh/NCKH-rebuild"
REPO_DIR = "/content/MoodNote-AI"
DRIVE_DIR = "/content/drive/MyDrive/MoodNote-AI/ablation"

if not torch.cuda.is_available():
    raise RuntimeError("GPU not found. Runtime -> Change runtime type -> T4 GPU")
print(f"GPU: {torch.cuda.get_device_name(0)}")

drive.mount("/content/drive")

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(
        ["git", "clone", "-b", BRANCH, "https://github.com/MoodNote/MoodNote-AI.git", REPO_DIR],
        check=True,
    )
os.chdir(REPO_DIR)

# Run results and saved models go to Drive. Results already committed to git are copied there first.
for local, remote in (("reports/ablation", f"{DRIVE_DIR}/reports"), ("models/ablation", f"{DRIVE_DIR}/models")):
    os.makedirs(remote, exist_ok=True)
    if not os.path.islink(local):
        if os.path.isdir(local):
            shutil.copytree(local, remote, dirs_exist_ok=True)
            shutil.rmtree(local)
        os.makedirs(os.path.dirname(local), exist_ok=True)
        os.symlink(remote, local)

In [ ]:
# ── Cell 2: Dependencies + W&B ───────────────────────────────────────────────
# Versions match requirements.txt; pyvi segments the VSMEC text like the local pipeline.
!pip install -q --progress-bar off transformers==5.3.0 accelerate==1.13.0 scikit-learn==1.8.0 pyvi==0.1.1 wandb==0.25.0

try:
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    os.environ["WANDB_MODE"] = "offline"
    print("No WANDB_API_KEY secret: W&B logs offline")

In [ ]:
# ── Cell 3: Data (UIT-VSMEC download + segmentation, ablation train/val files) ─
!python -m src.data.real.download_vsmec
!python -m src.data.real.preprocess
!python -m src.data.ablation

## Ablation runs

One JSON per run in `reports/ablation/<scenario>_seed<S>.json`. To split the work across sessions, run
one scenario or seed at a time, e.g. `--scenario combined --seed 43`.

In [ ]:
# ── Cell 4: Train + evaluate (re-run after a disconnect: finished runs are skipped) ─
!python -m src.training.ablation_runner

In [ ]:
# ── Cell 5: Comparison ───────────────────────────────────────────────────────
from IPython.display import Markdown, display

display(Markdown(open("reports/ablation/comparison.md", encoding="utf-8").read()))

## Next steps (local machine)

1. Download `MyDrive/MoodNote-AI/ablation/reports/` into `reports/ablation/` and commit the JSON files
   (`comparison.md` is gitignored like every `*.md`; it is regenerated from the JSONs).
2. The seed-42 models stay on Drive (`MyDrive/MoodNote-AI/ablation/models/<scenario>/`) for phase 5.